# LLM-as-a-Judge Patterns (Building Robust Rubrics, Self-Consistency, and Grading Agents)
When an agent's final output or intermediate reasoning is open-ended natural language, deterministic code assertions (assert text == "expected") fail. To evaluate semantic correctness, helpfulness, tone, and reasoning quality at scale, modern AI engineering relies on LLM-as-a-Judge patterns.

In this topic, we examine how to use a more powerful model (or a specialized prompt) to grade agent trajectories and outputs reliably.

## 1. The Core Architecture of LLM-as-a-Judge
Instead of hardcoding rules, you design an evaluation prompt where an LLM is given three things:

The Context / User Prompt: What was asked of the agent?

The Agent's Execution Trace / Output: What did the agent do or say?

A Structured Rubric: Clear grading criteria, scoring scales (e.g., 1 to 5, or Pass/Fail), and definitions for each score.

The judge model returns a structured JSON object containing a score, a rationale, and feedback.

[User Prompt + Agent Trace] ──► [LLM Judge Model] ──► [Structured Score + Rationale (JSON)]
                                         ▲
                                         │
                             [Strict Evaluation Rubric]


## 2. Best Practices for Designing Reliable LLM Judges
LLM judges are prone to biases (like favoring longer responses or being overly lenient). To build an enterprise-grade judge, follow these four patterns:

### A. Use Structured Outputs (JSON Mode)
Never ask a judge model for a free-form paragraph score. Force the model to output a strict JSON schema containing separate fields for score, reasoning, and category_breakdowns. This makes programmatic ingestion and CI/CD pass/fail gates trivial.

### B. Provide Few-Shot Examples in the Rubric
Instead of just describing what a "Score of 5" looks like, provide concrete examples of a Good Trace and a Bad Trace directly inside the judge prompt. Few-shot calibration drastically reduces variance across judge runs.

### C. Criteria Decomposition (Multi-Dimensional Grading)
Don't ask a single prompt to judge "Is this agent good?" Break evaluation down into distinct sub-rubrics:

Factual Correctness: Did the agent hallucinate any data?

Tool Relevance: Did it use the right tools efficiently?

Constraint Adherence: Did it follow negative constraints (e.g., "Do not mention competitor X")?

D. Chain-of-Thought (CoT) Prompting for the Judge
Always instruct the judge model to write out its step-by-step reasoning before giving the final numerical score. This forces the model to analyze the evidence critically before committing to a grade.

## 3. Hands-On Python Pattern: Building an LLM-as-a-Judge Evaluator
Below is a Python pattern utilizing the official Google GenAI SDK (google-genai) to implement a structured LLM-as-a-Judge evaluator for an agent's final output.

In [ ]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize GenAI Client (assumes GEMINI_API_KEY is set in environment)
client = genai.Client()

# --- 1. Define Structured Output Schema for the Judge ---
class AgentEvaluationVerdict(BaseModel):
    score: int = Field(description="Score from 1 to 5, where 5 is exceptional.")
    passed: bool = Field(description="True if score >= 4, else False.")
    reasoning: str = Field(description="Step-by-step justification for the score based on the rubric.")
    hallucination_detected: bool = Field(description="True if the agent made up unverified facts.")

# --- 2. LLM-as-a-Judge Function ---
def evaluate_agent_output_with_llm(user_prompt: str, agent_output: str) -> AgentEvaluationVerdict:
    judge_system_instruction = """
    You are an expert AI Safety and Quality Assurance Judge. Your job is to evaluate 
    an autonomous agent's final response against the user prompt.
    
    Grading Rubric:
    - Score 5: Fully answers the prompt, factually accurate, clear, and professional.
    - Score 3: Partially answers the prompt or contains minor stylistic flaws, but no hallucinations.
    - Score 1: Fails to answer the prompt, contains severe hallucinations, or violates core instructions.
    
    Evaluate objectively and provide structured feedback.
    """

    prompt = f"""
    User Prompt: {user_prompt}
    Agent Final Output: {agent_output}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=judge_system_instruction,
            response_mime_type="application/json",
            response_schema=AgentEvaluationVerdict,
            temperature=0.0 # Zero temperature for deterministic judging consistency
        ),
    )
    
    # Parse the structured JSON response into our Pydantic model
    return AgentEvaluationVerdict.model_validate_json(response.text)

# --- Run Evaluation Test ---
if __name__ == "__main__":
    user_query = "What is the capital of France and its approximate population?"
    simulated_agent_response = "The capital of France is Paris. Its approximate population is around 2.1 million people within city limits."

    verdict = evaluate_agent_output_with_llm(user_query, simulated_agent_response)
    
    print("--- LLM Judge Verdict ---")
    print(f"Passed: {verdict.passed}")
    print(f"Score: {verdict.score}/5")
    print(f"Reasoning: {verdict.reasoning}")
    print(f"Hallucination Detected: {verdict.hallucination_detected}")